In [38]:
import json
import os

points_json = "/home/jovyan/work/data/output/points_gx_output.json"
stops_json = "/home/jovyan/work/data/output/stops_gx_output.json"
report_file = "/home/jovyan/work/data/output/validation_report.html"

with open(points_json, 'r', encoding='utf-8') as f:
    data_points = json.load(f)

with open(stops_json, 'r', encoding='utf-8') as f:
    data_stops = json.load(f)

results_points = data_points['validation_results'][0]
results_stops = data_stops['validation_results'][0]

statistics_points = results_points['statistics']
expectations_points = results_points['expectations']

statistics_stops = results_stops['statistics']
expectations_stops = results_stops['expectations']

In [39]:
def render_row(e):
    col = e['kwargs'].get('column', '-')
    stat = 'OK' if e['success'] else 'FAIL'
    stat_class = 'success' if e['success'] else 'danger'
    result = e.get('result', {})
    unexpected_percent = result.get('unexpected_percent', result.get('unexpected_percent_total', ''))
    unexpected_count = result.get('unexpected_count', '')
    elem_count = result.get('element_count', '')
    partial_unexpected = result.get('partial_unexpected_list', '')
    if isinstance(partial_unexpected, list):
        partial_unexpected = ', '.join([str(x) for x in partial_unexpected[:5]]) if partial_unexpected else ''
    extra = ''
    if e['expectation_type'] == "expect_column_values_to_be_of_type":
        extra = f"Observed: {result.get('observed_value')} (expected: {e['kwargs'].get('type_', '')})"
    if e['expectation_type'] == "expect_column_values_to_be_in_set":
        extra = f"unexpected = {partial_unexpected}"

    return f"""
        <tr class="table-{stat_class}">
            <td class="{stat_class}">{stat}</td>
            <td>{e['expectation_type']}</td>
            <td>{col}</td>
            <td>{str(e['success'])}</td>
            <td>{elem_count}</td>
            <td>{unexpected_count}</td>
            <td>{unexpected_percent if unexpected_percent != '' else '-'}</td>
            <td>{extra}</td>
        </tr>
    """

html = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <title>Validation Report</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {{
            background-color: #f8f9fa;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }}
        h1 {{
            text-align: center;
            color: #0d6efd;
            font-weight: 700;
        }}
        h2 {{
            color: #0d6efd;
            border-bottom: 2px solid #0d6efd;
            padding-bottom: 4px;
        }}
        .card {{
            margin-top: 2rem;
            border-radius: 10px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}
        table {{
            font-size: 0.9rem;
        }}
        .table-hover tbody tr:hover {{
            background-color: #f1f1f1;
        }}
        .success {{
            color: #198754;
            font-weight: bold;
        }}
        .danger {{
            color: #dc3545;
            font-weight: bold;
        }}
        .badge-success {{
            background-color: #198754;
        }}
        .badge-danger {{
            background-color: #dc3545;
        }}
        .status-badge {{
            font-size: 0.8rem;
            padding: 0.4em 0.7em;
        }}
        .section-img {{
            border-radius: 8px;
            border: 1px solid #dee2e6;
            max-width: 500px;
        }}
    </style>
</head>
<body>
<div class="container my-5">
    <h1>Colletivo Data<br><small class="text-muted">Validação de Trajetos e Paradas</small></h1>

    <!-- POINTS -->
    <div class="card p-4 bg-white">
        <h2>Points (Pontos de Trajeto)</h2>
        <p class="lead mb-1">
            <strong>Success Percent:</strong> {round(statistics_points['success_percent'], 2)}%
        </p>
        <p>
            Status:
            <span class="badge status-badge {'badge-success' if data_points['success'] else 'badge-danger'}">
                {'SUCCESS' if data_points['success'] else 'FAILED'}
            </span>
        </p>

        <div class="text-center my-4">
            <img src="imagem_points.png" alt="Points" class="img-fluid section-img">
        </div>

        <h5 class="mt-4 mb-3">Detailed Expectations</h5>
        <div class="table-responsive">
            <table class="table table-bordered table-hover align-middle">
                <thead class="table-light">
                    <tr>
                        <th>Status</th>
                        <th>Expectation Type</th>
                        <th>Column</th>
                        <th>Success</th>
                        <th>Element Count</th>
                        <th>Unexpected Count</th>
                        <th>Unexpected %</th>
                        <th>Details</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join([render_row(e) for e in expectations_points])}
                </tbody>
            </table>
        </div>
    </div>

    <!-- STOPS -->
    <div class="card p-4 bg-white mt-5">
        <h2>Stops (Paradas)</h2>
        <p class="lead mb-1">
            <strong>Success Percent:</strong> {round(statistics_stops['success_percent'], 2)}%
        </p>
        <p>
            Status:
            <span class="badge status-badge {'badge-success' if data_stops['success'] else 'badge-danger'}">
                {'SUCCESS' if data_stops['success'] else 'FAILED'}
            </span>
        </p>

        <div class="text-center my-4">
            <img src="imagem_stops.png" alt="Stops" class="img-fluid section-img">
        </div>

        <h5 class="mt-4 mb-3">Detailed Expectations</h5>
        <div class="table-responsive">
            <table class="table table-bordered table-hover align-middle">
                <thead class="table-light">
                    <tr>
                        <th>Status</th>
                        <th>Expectation Type</th>
                        <th>Column</th>
                        <th>Success</th>
                        <th>Element Count</th>
                        <th>Unexpected Count</th>
                        <th>Unexpected %</th>
                        <th>Details</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join([render_row(e) for e in expectations_stops])}
                </tbody>
            </table>
        </div>
    </div>

    <footer class="text-center text-muted mt-5">
        <small>Gerado automaticamente por Colletivo Data - {os.getenv('USER', 'pipeline')}</small>
    </footer>

</div>
<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
"""

# Salva o HTML
os.makedirs(os.path.dirname(report_file), exist_ok=True)
with open(report_file, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Relatório salvo em: {report_file}")


Relatório salvo em: /home/jovyan/work/data/output/validation_report.html
